# 05. CNN Gradient x Input

embedding 기준 Gradient x Input attribution을 확인하고 unigram occlusion과 비교한다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd()
if (cwd / 'xai_outputs').exists(): OUT = cwd / 'xai_outputs'
elif (cwd.parent / 'xai_outputs').exists(): OUT = cwd.parent / 'xai_outputs'
else: OUT = Path('XAI/CNN/xai_outputs')

selected = pd.read_csv(OUT / 'cnn_xai_selected_samples.csv')
grad = pd.read_csv(OUT / 'cnn_gradient_x_input.csv')
uni = pd.read_csv(OUT / 'cnn_unigram_occlusion.csv')
print(OUT.resolve(), len(grad))

In [ ]:
sample_id = 'custom_1' if 'custom_1' in set(selected['sample_id']) else selected['sample_id'].iloc[0]
sample = selected[selected['sample_id'] == sample_id].iloc[0]
g = grad[grad['sample_id'] == sample_id].sort_values('normalized_abs_score', ascending=False)
u = uni[uni['sample_id'] == sample_id].sort_values('prob_drop', ascending=False)

print('sample_id:', sample_id)
print('text:', sample['text'])
display(g[['position', 'token', 'signed_score', 'abs_score', 'normalized_abs_score']].head(15))

In [ ]:
merged = g[['position', 'token', 'normalized_abs_score']].merge(
    u[['position', 'prob_drop']], on='position', how='left'
).sort_values('position')
display(merged)

ax = merged.plot(x='token', y=['normalized_abs_score', 'prob_drop'], kind='bar', figsize=(10, 4))
ax.set_title(f'Gradient x Input vs Unigram Occlusion: {sample_id}')
ax.set_xlabel('token')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()